In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

%cd /content/drive/MyDrive/geometric-data-analysis/tgn

!ls

/content/drive/MyDrive/geometric-data-analysis/tgn
CONTRIBUTING.md  LICENSE      README.md		 testing.py
data		 log	      requirements.txt	 train_self_supervised.py
evaluation	 model	      results		 train_supervised.py
figures		 modules      saved_checkpoints  utils
__init__.py	 __pycache__  saved_models


In [4]:
# Install Python 3.8
!sudo apt-get update -y
!sudo apt-get install python3.8 python3.8-venv python3.8-dev -y

!python3.8 -m venv /content/py38env

# Activate and install packages
!/content/py38env/bin/pip install --upgrade pip
!/content/py38env/bin/pip install -r requirements.txt


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,196 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,286 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,841 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/r

In [5]:
!/content/py38env/bin/python --version

Python 3.8.20


In [6]:
import torch
print(f"GPU: {torch.cuda.is_available()}")

GPU: True


In [ ]:
!/content/py38env/bin/python train_self_supervised.py --use_memory --prefix tgn-attn --n_runs 10


INFO:root:Namespace(aggregator='last', backprop_every=1, bs=200, data='wikipedia', different_new_nodes=False, drop_out=0.1, dyrep=False, embedding_module='graph_attention', gpu=0, lr=0.0001, memory_dim=172, memory_update_at_end=False, memory_updater='gru', message_dim=100, message_function='identity', n_degree=10, n_epoch=50, n_head=2, n_layer=1, n_runs=10, node_dim=100, patience=5, prefix='tgn-attn', randomize_features=False, time_dim=100, uniform=False, use_destination_embedding_in_message=False, use_memory=True, use_source_embedding_in_message=False)
The dataset has 157474 interactions, involving 9227 different nodes
The training dataset has 81029 interactions, involving 6141 different nodes
The validation dataset has 23621 interactions, involving 3256 different nodes
The test dataset has 23621 interactions, involving 3564 different nodes
The new node validation dataset has 12016 interactions, involving 2120 different nodes
The new node test dataset has 11715 interactions, involving

In [ ]:
!/content/py38env/bin/python -c "import torch; print(torch.__version__)"


1.6.0


In [ ]:
import pickle
import numpy as np

test_ap_list = []
new_node_ap_list = []
epoch_time_means = []

# ---- 1) Charger tgn-attn.pkl (run 0) ----
base = "results/tgn-attn.pkl"
with open(base, 'rb') as f:
    data = pickle.load(f)
    test_ap_list.append(data["test_ap"])
    new_node_ap_list.append(data["new_node_test_ap"])
    epoch_time_means.append(np.mean(data["epoch_times"]))
    print(f"[OK] {base} -> test_ap={data['test_ap']}, new_node_test_ap={data['new_node_test_ap']}, mean_epoch={np.mean(data['epoch_times']):.4f}")

# ---- 2) Charger tgn-attn_1.pkl à tgn-attn_9.pkl ----
for i in range(1, 10):
    file = f"results/tgn-attn_{i}.pkl"
    try:
        with open(file, 'rb') as f:
            data = pickle.load(f)
            test_ap_list.append(data["test_ap"])
            new_node_ap_list.append(data["new_node_test_ap"])
            epoch_time_means.append(np.mean(data["epoch_times"]))

            print(f"[OK] {file} -> test_ap={data['test_ap']}, new_node_test_ap={data['new_node_test_ap']}, mean_epoch={np.mean(data['epoch_times']):.4f}")
    except Exception as e:
        print(f"[ERROR] {file}: {e}")

print("\n==================== FINAL RESULTS ====================\n")
print(f"Test_AP        → Mean = {np.mean(test_ap_list):.4f}, Std = {np.std(test_ap_list):.4f}")
print(f"New_Node_AP    → Mean = {np.mean(new_node_ap_list):.4f}, Std = {np.std(new_node_ap_list):.4f}")
print(f"Epoch_times    → Mean = {np.mean(epoch_time_means):.4f}")


[OK] results/tgn-attn.pkl -> test_ap=0.9848135075910786, new_node_test_ap=0.9771293505834222, mean_epoch=22.1671
[OK] results/tgn-attn_1.pkl -> test_ap=0.9843561097535034, new_node_test_ap=0.976353634877542, mean_epoch=22.2147
[OK] results/tgn-attn_2.pkl -> test_ap=0.984383686758018, new_node_test_ap=0.9778704160864641, mean_epoch=22.2190
[OK] results/tgn-attn_3.pkl -> test_ap=0.9850717578056654, new_node_test_ap=0.9756979198481189, mean_epoch=22.3814
[OK] results/tgn-attn_4.pkl -> test_ap=0.9852441622523991, new_node_test_ap=0.9800642778727154, mean_epoch=22.1515
[OK] results/tgn-attn_5.pkl -> test_ap=0.9853717081285132, new_node_test_ap=0.9775701383132145, mean_epoch=21.9390
[OK] results/tgn-attn_6.pkl -> test_ap=0.9856784188416758, new_node_test_ap=0.9789081725475937, mean_epoch=21.9425
[OK] results/tgn-attn_7.pkl -> test_ap=0.9847002357574179, new_node_test_ap=0.9773505141811948, mean_epoch=21.8810
[OK] results/tgn-attn_8.pkl -> test_ap=0.9852053643042857, new_node_test_ap=0.978046

In [ ]:
!/content/py38env/bin/python train_supervised.py \
    --use_memory \
    --prefix tgn-attn \
    --n_runs 10 \
    --data wikipedia

INFO:root:Namespace(aggregator='last', backprop_every=1, bs=100, data='wikipedia', different_new_nodes=False, drop_out=0.1, embedding_module='graph_attention', gpu=0, lr=0.0003, memory_dim=172, memory_update_at_end=False, message_dim=100, message_function='identity', n_degree=10, n_epoch=10, n_head=2, n_layer=1, n_neg=1, n_runs=10, new_node=False, node_dim=100, patience=5, prefix='tgn-attn', randomize_features=False, time_dim=100, uniform=False, use_destination_embedding_in_message=False, use_memory=True, use_source_embedding_in_message=False, use_validation=False)
DEBUG:root:Num of training instances: 133853
DEBUG:root:Num of batches per epoch: 1339
INFO:root:Loading saved TGN model
INFO:root:TGN models loaded
INFO:root:Start training node classification task
INFO:root:Epoch 0: train loss: 0.038731487244905986, val auc: 0.8887383505496498, time: 58.812891483306885
INFO:root:Epoch 1: train loss: 0.01031119231314797, val auc: 0.8846376669095845, time: 60.076598167419434
INFO:root:Epoch 

In [ ]:
!/content/py38env/bin/python -c "import torch; print('CUDA available:', torch.cuda.is_available()); print('PyTorch version:', torch.__version__)"


CUDA available: False
PyTorch version: 1.6.0


In [ ]:
import pickle
import numpy as np

test_ap_list = []

# ---- 1) Charger tgn-attn.pkl (run 0) ----
base = "results/tgn-attn_node_classification.pkl"
with open(base, 'rb') as f:
    data = pickle.load(f)
    print(data.keys())
    test_ap_list.append(data["test_ap"])
    print(f"[OK] {base} -> test_ap={data['test_ap']}, new_node_test_ap={data['new_node_test_ap']}, mean_epoch={np.mean(data['epoch_times']):.4f}")

# ---- 2) Charger tgn-attn_1.pkl à tgn-attn_9.pkl ----
for i in range(1, 10):
    file = f"results/tgn-attn_node_classification_{i}.pkl"
    try:
        with open(file, 'rb') as f:
            data = pickle.load(f)
            test_ap_list.append(data["test_ap"])

            print(f"[OK] {file} -> test_ap={data['test_ap']}, new_node_test_ap={data['new_node_test_ap']}, mean_epoch={np.mean(data['epoch_times']):.4f}")
    except Exception as e:
        print(f"[ERROR] {file}: {e}")

print("\n==================== FINAL RESULTS ====================\n")
print(f"Test_AP        → Mean = {np.mean(test_ap_list):.4f}, Std = {np.std(test_ap_list):.4f}")


dict_keys(['val_aps', 'test_ap', 'train_losses', 'epoch_times', 'new_nodes_val_aps', 'new_node_test_ap'])
[OK] results/tgn-attn_node_classification.pkl -> test_ap=0.8919695427361798, new_node_test_ap=0, mean_epoch=0.0000
[OK] results/tgn-attn_node_classification_1.pkl -> test_ap=0.8901900735308294, new_node_test_ap=0, mean_epoch=0.0000
[OK] results/tgn-attn_node_classification_2.pkl -> test_ap=0.8982323874962888, new_node_test_ap=0, mean_epoch=0.0000
[OK] results/tgn-attn_node_classification_3.pkl -> test_ap=0.897319517866025, new_node_test_ap=0, mean_epoch=0.0000
[OK] results/tgn-attn_node_classification_4.pkl -> test_ap=0.8960663705383134, new_node_test_ap=0, mean_epoch=0.0000
[OK] results/tgn-attn_node_classification_5.pkl -> test_ap=0.8958918938719167, new_node_test_ap=0, mean_epoch=0.0000
[OK] results/tgn-attn_node_classification_6.pkl -> test_ap=0.8960572129232263, new_node_test_ap=0, mean_epoch=0.0000
[OK] results/tgn-attn_node_classification_7.pkl -> test_ap=0.8907057918541569,

In [ ]:
! /content/py38env/bin/python train_self_supervised.py --use_memory --embedding_module identity --prefix tgn-id --n_runs 10

INFO:root:Namespace(aggregator='last', backprop_every=1, bs=200, data='wikipedia', different_new_nodes=False, drop_out=0.1, dyrep=False, embedding_module='identity', gpu=0, lr=0.0001, memory_dim=172, memory_update_at_end=False, memory_updater='gru', message_dim=100, message_function='identity', n_degree=10, n_epoch=50, n_head=2, n_layer=1, n_runs=10, node_dim=100, patience=5, prefix='tgn-id', randomize_features=False, time_dim=100, uniform=False, use_destination_embedding_in_message=False, use_memory=True, use_source_embedding_in_message=False)
The dataset has 157474 interactions, involving 9227 different nodes
The training dataset has 81029 interactions, involving 6141 different nodes
The validation dataset has 23621 interactions, involving 3256 different nodes
The test dataset has 23621 interactions, involving 3564 different nodes
The new node validation dataset has 12016 interactions, involving 2120 different nodes
The new node test dataset has 11715 interactions, involving 2437 dif

In [ ]:
import pickle
import numpy as np

test_ap_list = []
new_node_ap_list = []
epoch_time_means = []

# ---- 1) Charger tgn-attn.pkl (run 0) ----
base = "results/tgn-id.pkl"
with open(base, 'rb') as f:
    data = pickle.load(f)
    test_ap_list.append(data["test_ap"])
    new_node_ap_list.append(data["new_node_test_ap"])
    epoch_time_means.append(np.mean(data["epoch_times"]))
    print(f"[OK] {base} -> test_ap={data['test_ap']}, new_node_test_ap={data['new_node_test_ap']}, mean_epoch={np.mean(data['epoch_times']):.4f}")

# ---- 2) Charger tgn-attn_1.pkl à tgn-attn_9.pkl ----
for i in range(1, 10):
    file = f"results/tgn-id_{i}.pkl"
    try:
        with open(file, 'rb') as f:
            data = pickle.load(f)
            test_ap_list.append(data["test_ap"])
            new_node_ap_list.append(data["new_node_test_ap"])
            epoch_time_means.append(np.mean(data["epoch_times"]))

            print(f"[OK] {file} -> test_ap={data['test_ap']}, new_node_test_ap={data['new_node_test_ap']}, mean_epoch={np.mean(data['epoch_times']):.4f}")
    except Exception as e:
        print(f"[ERROR] {file}: {e}")

print("\n==================== FINAL RESULTS ====================\n")
print(f"Test_AP        → Mean = {np.mean(test_ap_list):.4f}, Std = {np.std(test_ap_list):.4f}")
print(f"New_Node_AP    → Mean = {np.mean(new_node_ap_list):.4f}, Std = {np.std(new_node_ap_list):.4f}")
print(f"Epoch_times    → Mean = {np.mean(epoch_time_means):.4f}")


[OK] results/tgn-id.pkl -> test_ap=0.9540317691768322, new_node_test_ap=0.9300359779363658, mean_epoch=18.7614
[OK] results/tgn-id_1.pkl -> test_ap=0.956115549271963, new_node_test_ap=0.929231486469521, mean_epoch=18.8453
[OK] results/tgn-id_2.pkl -> test_ap=0.9597790693480378, new_node_test_ap=0.9350288167483725, mean_epoch=18.0058
[OK] results/tgn-id_3.pkl -> test_ap=0.9598549391577683, new_node_test_ap=0.9387386367437445, mean_epoch=17.1096
[OK] results/tgn-id_4.pkl -> test_ap=0.9562807892778565, new_node_test_ap=0.9303536913274929, mean_epoch=16.5897
[OK] results/tgn-id_5.pkl -> test_ap=0.9568972716171248, new_node_test_ap=0.9279480744847327, mean_epoch=16.5562
[OK] results/tgn-id_6.pkl -> test_ap=0.9558739011633526, new_node_test_ap=0.9306378629960519, mean_epoch=16.7885
[OK] results/tgn-id_7.pkl -> test_ap=0.9595420381110231, new_node_test_ap=0.9339908542248968, mean_epoch=16.8156
[OK] results/tgn-id_8.pkl -> test_ap=0.9594058874917761, new_node_test_ap=0.933994356024235, mean_ep

In [ ]:
# TGN-sum
! /content/py38env/bin/python train_self_supervised.py --use_memory --embedding_module graph_sum --prefix tgn-sum --n_runs 10

INFO:root:Namespace(aggregator='last', backprop_every=1, bs=200, data='wikipedia', different_new_nodes=False, drop_out=0.1, dyrep=False, embedding_module='graph_sum', gpu=0, lr=0.0001, memory_dim=172, memory_update_at_end=False, memory_updater='gru', message_dim=100, message_function='identity', n_degree=10, n_epoch=50, n_head=2, n_layer=1, n_runs=10, node_dim=100, patience=5, prefix='tgn-sum', randomize_features=False, time_dim=100, uniform=False, use_destination_embedding_in_message=False, use_memory=True, use_source_embedding_in_message=False)
The dataset has 157474 interactions, involving 9227 different nodes
The training dataset has 81029 interactions, involving 6141 different nodes
The validation dataset has 23621 interactions, involving 3256 different nodes
The test dataset has 23621 interactions, involving 3564 different nodes
The new node validation dataset has 12016 interactions, involving 2120 different nodes
The new node test dataset has 11715 interactions, involving 2437 d